In [ ]:
!pip install pypdf python-docx gradio

import re
import os
import numpy as np
import pandas as pd

from pypdf import PdfReader
from docx import Document

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import gradio as gr


job_roles = {

    "Data Scientist": {
        "description": """
        python machine learning data science statistics pandas numpy
        scikit learn tensorflow pytorch deep learning data visualization
        matplotlib seaborn sql artificial intelligence nlp
        natural language processing predictive modeling
        """
    },

    "Machine Learning Engineer": {
        "description": """
        python machine learning artificial intelligence tensorflow pytorch
        scikit learn deep learning neural networks nlp computer vision
        model deployment docker kubernetes mlops pandas numpy
        """
    },

    "Data Analyst": {
        "description": """
        sql excel python pandas numpy data analysis statistics
        tableau power bi data visualization matplotlib seaborn
        business intelligence dashboards reporting
        """
    },

    "Python Developer": {
        "description": """
        python django flask fastapi rest api oop object oriented programming
        sql postgresql mysql git github docker backend development
        """
    },

    "Web Developer": {
        "description": """
        html css javascript react angular vue nodejs express
        frontend backend web development rest api git github
        """
    },

    "Full Stack Developer": {
        "description": """
        html css javascript react nodejs express python django
        flask sql mongodb mysql postgresql rest api
        frontend backend full stack git github docker
        """
    },

    "Software Developer": {
        "description": """
        python java c++ javascript data structures algorithms
        object oriented programming sql git github software development
        debugging testing api
        """
    },

    "AI Engineer": {
        "description": """
        artificial intelligence machine learning deep learning
        python tensorflow pytorch neural networks nlp computer vision
        generative ai transformers llm natural language processing
        """
    },

    "NLP Engineer": {
        "description": """
        python nlp natural language processing machine learning
        deep learning transformers bert text classification
        sentiment analysis tokenization spacy nltk tensorflow pytorch
        """
    },

    "DevOps Engineer": {
        "description": """
        linux docker kubernetes aws azure cloud computing
        jenkins git github cicd continuous integration
        continuous deployment terraform ansible networking
        """
    },

    "Cloud Engineer": {
        "description": """
        aws azure google cloud cloud computing docker kubernetes
        linux networking terraform devops serverless
        """
    },

    "Cybersecurity Analyst": {
        "description": """
        cybersecurity network security ethical hacking penetration testing
        linux python cryptography vulnerability assessment
        security monitoring firewalls incident response
        """
    }
}

skills_database = [

    "python", "java", "c++", "c", "javascript", "typescript",
    "r", "php", "ruby", "go", "kotlin", "swift",

    "html", "css", "react", "angular", "vue", "nodejs",
    "express", "django", "flask", "fastapi",

    "sql", "mysql", "postgresql", "mongodb", "excel",
    "pandas", "numpy", "matplotlib", "seaborn",
    "tableau", "power bi", "statistics",

    "machine learning", "deep learning", "artificial intelligence",
    "tensorflow", "pytorch", "scikit learn", "scikit-learn",
    "keras", "nlp", "natural language processing",
    "computer vision", "transformers", "bert",
    "generative ai", "llm",

    "docker", "kubernetes", "aws", "azure", "google cloud",
    "linux", "jenkins", "terraform", "ansible",
    "git", "github", "cicd",

    "cybersecurity", "ethical hacking", "penetration testing",
    "network security", "cryptography",

    "rest api", "api", "data structures", "algorithms",
    "object oriented programming", "oop"
]

def extract_text_from_pdf(pdf_path):

    text = ""

    try:
        reader = PdfReader(pdf_path)

        for page in reader.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    except Exception as e:

        return f"ERROR: {str(e)}"

    return text



def extract_text_from_docx(docx_path):

    text = ""

    try:

        document = Document(docx_path)

        # Extract paragraphs
        for paragraph in document.paragraphs:

            if paragraph.text.strip():
                text += paragraph.text + "\n"

        # Extract tables
        for table in document.tables:

            for row in table.rows:

                for cell in row.cells:

                    if cell.text.strip():
                        text += cell.text + "\n"

    except Exception as e:

        return f"ERROR: {str(e)}"

    return text


def extract_text_from_resume(file_path):

    if file_path is None:

        return "ERROR: No file uploaded."

    extension = os.path.splitext(file_path)[1].lower()

    if extension == ".pdf":

        return extract_text_from_pdf(file_path)

    elif extension == ".docx":

        return extract_text_from_docx(file_path)

    else:

        return "ERROR: Unsupported file format. Please upload PDF or DOCX."



def clean_text(text):

    text = text.lower()

    # Replace special characters
    text = re.sub(r"[^a-zA-Z0-9+#.\- ]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()



def extract_skills(text):

    cleaned = clean_text(text)

    found_skills = []

    for skill in skills_database:

        skill_clean = skill.lower()

        pattern = r"(?<!\w)" + re.escape(skill_clean) + r"(?!\w)"

        if re.search(pattern, cleaned):

            found_skills.append(skill)

    return sorted(set(found_skills))



def predict_job_roles(resume_text):

    documents = [resume_text]

    role_names = []

    for role, data in job_roles.items():

        role_names.append(role)

        documents.append(data["description"])

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2)
    )

    vectors = vectorizer.fit_transform(documents)

    resume_vector = vectors[0]

    role_vectors = vectors[1:]

    similarities = cosine_similarity(
        resume_vector,
        role_vectors
    )[0]

    results = []

    for role, score in zip(role_names, similarities):

        results.append({

            "Job Role": role,

            "Match Score": round(
                float(score) * 100,
                2
            )

        })

    results = sorted(
        results,
        key=lambda x: x["Match Score"],
        reverse=True
    )

    return results



def extract_experience(text):

    patterns = [

        r"(\d+(?:\.\d+)?)\+?\s*(?:years|year)\s*(?:of)?\s*experience",

        r"experience\s*[:\-]?\s*(\d+(?:\.\d+)?)\+?\s*(?:years|year)",

        r"(\d+(?:\.\d+)?)\s*(?:years|year)\s*experience"

    ]

    text_lower = text.lower()

    for pattern in patterns:

        match = re.search(
            pattern,
            text_lower
        )

        if match:

            return match.group(1) + " years"

    return "Not detected"



def extract_education(text):

    education_keywords = [

        "b.tech",
        "btech",
        "b.e",
        "be ",
        "bachelor",

        "m.tech",
        "mtech",
        "m.e",
        "master",

        "mba",
        "mca",
        "bca",

        "b.sc",
        "bsc",

        "m.sc",
        "msc",

        "phd",

        "computer science",
        "information technology"

    ]

    text_lower = text.lower()

    found = []

    for keyword in education_keywords:

        if keyword in text_lower:

            found.append(keyword.upper())

    return sorted(set(found))



def find_missing_skills(
    resume_skills,
    predicted_role
):

    role_description = job_roles[
        predicted_role
    ]["description"]

    required_skills = []

    for skill in skills_database:

        if skill.lower() in role_description.lower():

            required_skills.append(skill)

    missing = [

        skill
        for skill in required_skills
        if skill not in resume_skills

    ]

    return sorted(set(missing))


def detect_sections(text):

    text_lower = text.lower()

    sections = {

        "Projects": [

            "project",
            "projects",
            "academic project",
            "personal project"

        ],

        "Internship": [

            "internship",
            "intern"

        ],

        "Experience": [

            "experience",
            "work experience",
            "professional experience"

        ],

        "Certifications": [

            "certification",
            "certifications",
            "certificate"

        ],

        "Achievements": [

            "achievement",
            "achievements",
            "award"

        ]

    }

    result = {}

    for section, keywords in sections.items():

        result[section] = any(

            keyword in text_lower

            for keyword in keywords

        )

    return result




def analyze_resume(resume_file):

    if resume_file is None:

        return (

            "Please upload a PDF or DOCX resume.",

            None,

            "",

            "",

            ""

        )

    # Extract PDF or DOCX
    text = extract_text_from_resume(
        resume_file
    )

    if text.startswith("ERROR"):

        return (

            text,

            None,

            "",

            "",

            ""

        )

    if len(text.strip()) < 50:

        return (

            "Could not extract enough text from the resume. "
            "Please upload a text-based PDF or DOCX.",

            None,

            "",

            "",

            ""

        )

    # Clean text
    cleaned = clean_text(text)

    # Skills
    resume_skills = extract_skills(cleaned)

    # Job prediction
    predictions = predict_job_roles(cleaned)

    # Top role
    top_role = predictions[0]["Job Role"]

    # Missing skills
    missing_skills = find_missing_skills(

        resume_skills,

        top_role

    )

    # Experience
    experience = extract_experience(text)

    # Education
    education = extract_education(text)

    # Sections
    sections = detect_sections(text)


 
    report = "# AI Resume Analysis\n\n"

    report += "## Recommended Job Role\n\n"

    report += f"### {top_role}\n\n"

    report += (

        f"**Match Score:** "
        f"{predictions[0]['Match Score']}%\n\n"

    )

    report += "## Top Job Role Predictions\n\n"

    report += "| Rank | Job Role | Match Score |\n"

    report += "|---:|---|---:|\n"

    for i, result in enumerate(
        predictions[:5],
        1
    ):

        report += (

            f"| {i} | {result['Job Role']} | "
            f"{result['Match Score']}% |\n"

        )


    # Skills
    report += "\n## Skills Detected\n\n"

    if resume_skills:

        for skill in resume_skills:

            report += f"- {skill}\n"

    else:

        report += (
            "No known technical skills detected.\n"
        )


    # Missing skills
    report += "\n## Skills You Could Add\n\n"

    if missing_skills:

        for skill in missing_skills[:10]:

            report += f"- {skill}\n"

    else:

        report += (
            "No major missing skills detected.\n"
        )


    # Education
    report += "\n## Education\n\n"

    if education:

        report += ", ".join(education)

    else:

        report += (
            "Education information not detected."
        )


    # Experience
    report += "\n\n## Experience\n\n"

    report += experience


    # Sections
    report += (
        "\n\n## Resume Sections Detected\n\n"
    )

    for section, detected in sections.items():

        status = (
            "Found"
            if detected
            else
            "Not Found"
        )

        report += (
            f"- **{section}:** {status}\n"
        )


    # Recommendations
    report += "\n## Recommendations\n\n"

    recommendations = []

    if len(resume_skills) < 5:

        recommendations.append(
            "Add more relevant technical skills "
            "to your resume."
        )

    if not sections["Projects"]:

        recommendations.append(
            "Add 2–3 strong projects with "
            "technologies and measurable results."
        )

    if not sections["Certifications"]:

        recommendations.append(
            "Consider adding relevant certifications "
            "or courses."
        )

    if missing_skills:

        recommendations.append(

            "Learn some of the missing skills "
            f"listed above for the {top_role} role."

        )

    if not recommendations:

        recommendations.append(

            "Your resume has a good technical "
            "skill coverage. Continue improving "
            "projects and measurable achievements."

        )

    for recommendation in recommendations:

        report += f"- {recommendation}\n"


    # Raw extracted text
    raw_text = text

    # DataFrame
    prediction_df = pd.DataFrame(
        predictions[:5]
    )

    return (

        report,

        prediction_df,

        ", ".join(resume_skills),

        raw_text,

        top_role

    )


with gr.Blocks(
    title="AI Resume Analyzer & Job Role Predictor"
) as app:

    gr.Markdown(
        """
        # AI Resume Analyzer & Job Role Predictor

        Upload your resume in **PDF or DOCX format**
        and the system will:

        - Extract resume text
        - Detect technical skills
        - Predict suitable job roles
        - Calculate role match scores
        - Identify missing skills
        - Give resume improvement recommendations
        """
    )


    with gr.Row():

        resume_input = gr.File(

            label="Upload Resume (PDF or DOCX)",

            file_types=[
                ".pdf",
                ".docx"
            ],

            type="filepath"

        )


        analyze_button = gr.Button(

            "Analyze Resume",

            variant="primary"

        )


    report_output = gr.Markdown(

        label="Resume Analysis"

    )


    with gr.Row():

        prediction_output = gr.Dataframe(

            headers=[
                "Job Role",
                "Match Score"
            ],

            label="Top Job Role Predictions",

            interactive=False

        )


    skills_output = gr.Textbox(

        label="Skills Detected",

        interactive=False

    )


    role_output = gr.Textbox(

        label="Recommended Job Role",

        interactive=False

    )


    with gr.Accordion(

        "View Extracted Resume Text",

        open=False

    ):

        raw_text_output = gr.Textbox(

            label="Extracted Text",

            lines=15,

            interactive=False

        )



    analyze_button.click(

        fn=analyze_resume,

        inputs=resume_input,

        outputs=[

            report_output,

            prediction_output,

            skills_output,

            raw_text_output,

            role_output

        ]

    )


app.launch(
    share=True,
    debug=True
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pypdf]
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://dbf30097c741427c22.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
